# Used Car Price Predictor

**Goal:** predict the selling price of a used car from its age, kilometres driven, fuel type, brand, etc.

**Plan:** load data → clean → EDA → feature engineering → compare 3 models → explain results.

> Replace the text in the *Your notes* cells with your own observations. Those notes are what make this a portfolio piece.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed: run `pip install xgboost`")

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
CURRENT_YEAR = 2026

## 2. Load data

**Get the real data (recommended):** download a CarDekho used-car CSV from Kaggle (search *"Vehicle dataset from CarDekho"*), save it as `data/cars.csv`.

If the file is missing, the cell below creates a **synthetic demo dataset** so you can run the whole notebook right away. Switch to real data before publishing.

In [ ]:
DATA_PATH = "data/cars.csv"
TARGET = "selling_price"   # change if your file names the price column differently

def make_demo_data(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    brands = ["Maruti", "Hyundai", "Honda", "Toyota", "Tata", "BMW", "Audi"]
    base = {"Maruti": 4, "Hyundai": 5, "Honda": 6, "Toyota": 9, "Tata": 5, "BMW": 25, "Audi": 27}  # lakh
    brand = rng.choice(brands, n, p=[.30, .22, .13, .10, .12, .07, .06])
    year = rng.integers(2005, 2025, n)
    age = 2026 - year
    km = np.clip(rng.normal(12000 * age + 8000, 15000), 1000, 300000).astype(int)
    fuel = rng.choice(["Petrol", "Diesel", "CNG"], n, p=[.55, .40, .05])
    trans = rng.choice(["Manual", "Automatic"], n, p=[.75, .25])
    owner = rng.choice(["First Owner", "Second Owner", "Third Owner"], n, p=[.7, .22, .08])
    price = np.array([base[b] for b in brand]) * 1e5
    price = price * np.exp(-0.09 * age) * (1 - km / 900000)
    price *= np.where(fuel == "Diesel", 1.15, 1.0) * np.where(trans == "Automatic", 1.2, 1.0)
    price *= np.where(owner == "First Owner", 1.0, np.where(owner == "Second Owner", 0.9, 0.8))
    price *= rng.lognormal(0, 0.12, n)
    name = [f"{b} Model{rng.integers(1,6)}" for b in brand]
    return pd.DataFrame({"name": name, "year": year, "selling_price": price.astype(int),
                         "km_driven": km, "fuel": fuel, "transmission": trans, "owner": owner})

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print("Loaded real data:", df.shape)
else:
    df = make_demo_data()
    print("⚠️ Using SYNTHETIC demo data:", df.shape)

df.head()

## 3. First look at the data

In [ ]:
df.info()
print()
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe()

## 4. Cleaning

Typical issues in real car data: duplicates, missing values, text mixed with numbers (like `"18.9 kmpl"`), and extreme outliers.

In [ ]:
df = df.drop_duplicates().copy()

# Convert columns like "18.9 kmpl", "1248 CC", "74 bhp" into numbers (only runs if those columns exist)
for col in ["mileage", "engine", "max_power"]:
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].str.extract(r"([\d.]+)")[0].astype(float)

# Drop rows with no target; fill numeric gaps with median, categorical gaps with mode
df = df.dropna(subset=[TARGET])
for col in df.columns:
    if df[col].isna().any():
        if df[col].dtype.kind in "if":
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

# Remove extreme outliers using the 1st and 99th percentile of price and km
for col in [TARGET, "km_driven"]:
    lo, hi = df[col].quantile([0.01, 0.99])
    df = df[(df[col] >= lo) & (df[col] <= hi)]

print("Shape after cleaning:", df.shape)

**Your notes:** what was messy? How many rows did you remove and why?

## 5. Feature engineering

In [ ]:
df["car_age"] = CURRENT_YEAR - df["year"]
df["brand"] = df["name"].str.split().str[0]

# Group rare brands so one-hot encoding doesn't explode
top_brands = df["brand"].value_counts().head(15).index
df["brand"] = df["brand"].where(df["brand"].isin(top_brands), "Other")

# Price is right-skewed, so models often do better on log(price)
df["log_price"] = np.log1p(df[TARGET])

df[["name", "brand", "year", "car_age", TARGET]].head()

## 6. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], bins=50, ax=axes[0]).set_title("Price distribution (skewed)")
sns.histplot(df["log_price"], bins=50, ax=axes[1]).set_title("log(price) distribution")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=df.sample(min(1500, len(df)), random_state=1), x="car_age", y=TARGET, alpha=.4, ax=axes[0])
axes[0].set_title("Price vs car age")
sns.scatterplot(data=df.sample(min(1500, len(df)), random_state=1), x="km_driven", y=TARGET, alpha=.4, ax=axes[1])
axes[1].set_title("Price vs km driven")
order = df.groupby("brand")[TARGET].median().sort_values().index
sns.boxplot(data=df, x="brand", y=TARGET, order=order, ax=axes[2], showfliers=False)
axes[2].tick_params(axis="x", rotation=60); axes[2].set_title("Price by brand")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap"); plt.show()

**Your notes:** what 3 things did the charts tell you? (e.g. *price drops fastest in the first 5 years; diesel cars cost more...*)

## 7. Prepare train / test sets

In [ ]:
drop_cols = [TARGET, "log_price", "name", "year"]   # year is replaced by car_age; name is replaced by brand
X = pd.get_dummies(df.drop(columns=drop_cols), drop_first=True)
y = df["log_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 8. Train and compare models

We train on `log(price)`, then convert predictions back to rupees so the metrics are easy to read.

In [ ]:
models = {"Linear Regression": LinearRegression(),
          "Random Forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)}
if HAS_XGB:
    models["XGBoost"] = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                                     subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, preds = [], {}

for name, model in models.items():
    cv_r2 = cross_val_score(model, X_train, y_train, cv=kf, scoring="r2").mean()
    model.fit(X_train, y_train)
    p = np.expm1(model.predict(X_test))          # back to rupees
    actual = np.expm1(y_test)
    preds[name] = p
    rows.append({"Model": name,
                 "CV R² (train)": round(cv_r2, 4),
                 "Test R²": round(r2_score(actual, p), 4),
                 "Test MAE": round(mean_absolute_error(actual, p)),
                 "Test RMSE": round(np.sqrt(mean_squared_error(actual, p)))})

results = pd.DataFrame(rows).set_index("Model")
results

**Your notes:** which model won and why do you think so? Did cross-validation R² agree with test R²?

## 9. Evaluate the best model

In [ ]:
best_name = results["Test RMSE"].idxmin()
best_model = models[best_name]
actual = np.expm1(y_test)
print("Best model:", best_name)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(actual, preds[best_name], alpha=.3)
lims = [actual.min(), actual.max()]
axes[0].plot(lims, lims, "r--"); axes[0].set_xlabel("Actual price"); axes[0].set_ylabel("Predicted price")
axes[0].set_title(f"{best_name}: predicted vs actual")

residuals = actual - preds[best_name]
sns.histplot(residuals, bins=50, ax=axes[1]); axes[1].set_title("Residuals (actual - predicted)")
plt.tight_layout(); plt.show()

## 10. What drives the price?

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
else:  # linear regression: use absolute coefficients
    imp = pd.Series(np.abs(best_model.coef_), index=X.columns).sort_values(ascending=False).head(12)

imp[::-1].plot(kind="barh", figsize=(8, 5), title=f"Top features ({best_name})")
plt.tight_layout(); plt.show()

## 11. Error analysis (this is what interviewers love)

In [ ]:
err = pd.DataFrame({"actual": actual, "predicted": preds[best_name]}, index=actual.index)
err["abs_pct_error"] = (err["actual"] - err["predicted"]).abs() / err["actual"] * 100
worst = err.join(df[["name", "year", "km_driven", "fuel"]]).sort_values("abs_pct_error", ascending=False).head(10)
print("Median % error:", round(err["abs_pct_error"].median(), 1), "%")
worst

**Your notes:** look at the worst predictions. Are they very old cars? Very expensive ones? Rare brands? Write down 2 or 3 patterns and what you'd do to fix them.

## 12. Save the model (optional)

In [ ]:
import joblib, os
os.makedirs("models", exist_ok=True)
joblib.dump({"model": best_model, "columns": list(X.columns)}, "models/car_price_model.pkl")
print("Saved.")

## 13. Summary (write this yourself)

- **Problem:** ...
- **Data:** ... rows, source, cleaning done
- **Best model:** ... with R² = ..., MAE = ₹...
- **Key drivers of price:** ...
- **Limitations:** ...
- **Next steps:** hyperparameter tuning, more features (city, variant), deploy as a Streamlit app